## Sync ops history from Lakebase to the Lakehouse

Reads `pdu_app.technician_alerts` and `pdu_app.unit_notes` from Lakebase (Postgres) and writes
them as Delta tables in `<catalog>.<ops_history_schema>` in Unity Catalog, with Delta Change Data
Feed enabled so downstream consumers can track incremental changes.

Run as the last task of the `pdu-refresh` job so the ops history tables in the Lakehouse are
always fresh after each pipeline cycle. The SQL alert `PDU technician alert` queries these tables.

Since both tables are typically small (operator actions, not telemetry), a full REPLACE on each
cycle is fast and avoids watermark tracking complexity.

In [0]:
# Job parameters - set by resources/refresh.job.yml via config.yml variables.
dbutils.widgets.text("catalog",           "mdk_demo_catalog")
dbutils.widgets.text("ops_history_schema","ops_history")
dbutils.widgets.text("lakebase_project",  "pdu-monitoring-dev")
dbutils.widgets.text("lakebase_database", "pdu_monitoring")

CATALOG            = dbutils.widgets.get("catalog")
OPS_SCHEMA         = dbutils.widgets.get("ops_history_schema")
LAKEBASE_PROJECT   = dbutils.widgets.get("lakebase_project")
LAKEBASE_DATABASE  = dbutils.widgets.get("lakebase_database")
APP_SCHEMA         = "pdu_app"  # always the same inside Lakebase

print(f"catalog={CATALOG}  ops_schema={OPS_SCHEMA}  project={LAKEBASE_PROJECT}  db={LAKEBASE_DATABASE}")

In [0]:
%pip install -q 'psycopg[binary]>=3'

In [0]:
import pandas as pd
import psycopg
from databricks.sdk import WorkspaceClient
from databricks.sdk.core import Config

w = WorkspaceClient()

# Resolve the Lakebase endpoint host from the SDK.
ep = w.postgres.get_endpoint(
    name=f"postgres_projects/{LAKEBASE_PROJECT}/branches/production/endpoints/primary"
)
LB_HOST = ep.host
print(f"Lakebase host: {LB_HOST}")

# Obtain an OAuth token for Lakebase using the job's identity.
cfg = Config()
auth_headers: dict = {}
cfg.authenticate()(auth_headers)                          # type: ignore[operator]
LB_TOKEN = auth_headers.get("Authorization", "").replace("Bearer ", "")

conn_params = dict(
    host=LB_HOST,
    port=5432,
    dbname=LAKEBASE_DATABASE,
    user="token",
    password=LB_TOKEN,
    sslmode="require",
)

with psycopg.connect(**conn_params) as conn:
    df_alerts = pd.DataFrame(
        conn.execute(
            "SELECT alert_id, unit_id, breach_started_ts, alerted_by, alerted_at, note "
            f"FROM {APP_SCHEMA}.technician_alerts ORDER BY alerted_at"
        ).fetchall(),
        columns=["alert_id", "unit_id", "breach_started_ts", "alerted_by", "alerted_at", "note"],
    )
    df_notes = pd.DataFrame(
        conn.execute(
            "SELECT note_id, unit_id, author, created_at, note "
            f"FROM {APP_SCHEMA}.unit_notes ORDER BY created_at"
        ).fetchall(),
        columns=["note_id", "unit_id", "author", "created_at", "note"],
    )

print(f"technician_alerts: {len(df_alerts)} rows  |  unit_notes: {len(df_notes)} rows")

In [0]:
# Create the ops_history schema if it does not exist.
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{OPS_SCHEMA}")

CDF_PROP = "'delta.enableChangeDataFeed' = 'true'"

def write_and_enable_cdf(df: pd.DataFrame, table: str) -> None:
    """Overwrite the UC Delta table and ensure CDF is on."""
    fqn = f"{CATALOG}.{OPS_SCHEMA}.{table}"
    (
        spark.createDataFrame(df)
        .write
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(fqn)
    )
    spark.sql(f"ALTER TABLE {fqn} SET TBLPROPERTIES ({CDF_PROP})")
    count = spark.table(fqn).count()
    print(f"{fqn}: {count} rows written, CDF enabled")

write_and_enable_cdf(df_alerts, "technician_alerts")
write_and_enable_cdf(df_notes,  "unit_notes")

print("Ops history sync complete.")